### 07 tuning + plant_part relations + evidence weights
reads: prep/, herb_compound_edges.csv, compound_target_labeled.csv  
writes: results/tune_*.csv, best_config.json

#### setup

In [1]:
import os, json, time, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score

DATA = 'C:/Users/Nimishavarma/OneDrive/Dokumenty'
PREP = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/prep'
OUT  = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/results'
CKPT = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/ckpt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH, EPOCHS, PATIENCE = 1024, 300, 20
TUNE_FOLDS = 3
K = 5
print(device)

cuda


#### load data

In [2]:
meta = json.load(open(f'{PREP}/meta.json'))
n_h, n_c, n_t = meta['n_h'], meta['n_c'], meta['n_t']
idm = json.load(open(f'{PREP}/id_maps.json'))
h2i, c2i, t2i = idm['h2i'], idm['c2i'], idm['t2i']

C_feat = torch.from_numpy(np.load(f'{PREP}/C_feat.npy')).to(device)
T_feat = torch.from_numpy(np.load(f'{PREP}/T_feat.npy')).to(device)
hc = np.load(f'{PREP}/hc_edges.npy')

ct_all = np.concatenate([np.load(f'{PREP}/{x}.npy') for x in ['ct_train','ct_val_pos','ct_test_pos']], axis=1)
pos_key = ct_all[0] * n_t + ct_all[1]
neg_key = np.setdiff1d(np.arange(n_c * n_t), pos_key)

known = torch.zeros(n_c, n_t, dtype=torch.bool, device=device)
known[torch.from_numpy(ct_all[0]).long(), torch.from_numpy(ct_all[1]).long()] = True
print(ct_all.shape, len(neg_key))

(2, 14166) 3095613


#### plant_part relations

In [3]:
# plant_part -> 6 groups, each group is its own herb-compound relation
PART = {'leaf': 'leaf',
        'rhizome': 'underground', 'root': 'underground', 'tuber': 'underground',
        'flower': 'flower_fruit_seed', 'fruit': 'flower_fruit_seed', 'seed': 'flower_fruit_seed',
        'bark': 'stem_bark', 'stem': 'stem_bark', 'wood': 'stem_bark',
        'aerial part': 'whole', 'whole plant': 'whole'}

raw = pd.read_csv(f'{DATA}/herb_compound_edges.csv', sep='\t', usecols=['herb_id','compound_id','plant_part'])
raw['grp'] = raw['plant_part'].fillna('').str.strip().str.lower().map(PART).fillna('other')
raw['h'] = raw['herb_id'].map(h2i)
raw['c'] = raw['compound_id'].map(c2i)
raw = raw.dropna(subset=['h','c']).drop_duplicates(['h','c','grp'])

part_names = sorted(raw['grp'].unique())
hc_parts = [raw.loc[raw['grp'] == g, ['h','c']].to_numpy().T.astype(np.int64) for g in part_names]
for g, e in zip(part_names, hc_parts):
    print(f'{g:20s} {e.shape[1]}')

flower_fruit_seed    1472
leaf                 1534
other                2294
stem_bark            703
underground          1270
whole                644


#### evidence weights

In [4]:
# evidence weight: how many databases (BindingDB, ChEMBL, NPASS) report the pair
ev = pd.read_csv(f'{DATA}/compound_target_labeled.csv', usecols=['compound_id','protein_id','evidence_source'])
ev['n_src'] = sum(ev['evidence_source'].fillna('').str.contains(db).astype(int) for db in ['BindingDB','ChEMBL','NPASS'])
ev['n_src'] = ev['n_src'].clip(lower=1)
ev['key'] = ev['compound_id'].map(c2i) * n_t + ev['protein_id'].map(t2i)

src_w = np.ones(n_c * n_t, dtype=np.float32)
src_w[ev['key'].to_numpy().astype(np.int64)] = ev['n_src'].to_numpy()
print(ev['n_src'].value_counts().sort_index())

n_src
1    8932
2    3882
3    1352
Name: count, dtype: int64


#### herb-target setup

In [5]:
H_df = pd.DataFrame(hc.T, columns=['h','c'])

def ht_keys(ct):
    j = H_df.merge(pd.DataFrame(ct.T, columns=['c','t']), on='c')
    return np.unique(j['h'].values * n_t + j['t'].values)

ht_all = ht_keys(ct_all)
ht_known = np.zeros(n_h * n_t, dtype=bool)
ht_known[ht_all] = True
ht_known = ht_known.reshape(n_h, n_t)
ht_neg = np.setdiff1d(np.arange(n_h * n_t), ht_all)
herb_cmps = [torch.from_numpy(np.unique(hc[1][hc[0] == h])).long().to(device) for h in range(n_h)]

#### model (number of relations depends on config)

In [6]:
class RelConv(nn.Module):
    def __init__(self, in_dim, out_dim, n_rel):
        super().__init__()
        self.W_rel  = nn.Parameter(torch.randn(n_rel, in_dim, out_dim) * 0.01)
        self.W_self = nn.Parameter(torch.randn(in_dim, out_dim) * 0.01)
        self.bias   = nn.Parameter(torch.zeros(out_dim))
    def forward(self, x, edge_sets):
        out = x @ self.W_self
        for r, (src, dst) in enumerate(edge_sets):
            if src.numel() == 0: continue
            m = x[src] @ self.W_rel[r]
            agg = torch.zeros(x.size(0), out.size(1), device=x.device, dtype=out.dtype)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(x.size(0), device=x.device)
            deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
            out = out + agg / deg.clamp(min=1).unsqueeze(1)
        return out + self.bias

class Model(nn.Module):
    def __init__(self, n_rel, d=128, dropout=0.2):
        super().__init__()
        self.herb_emb = nn.Embedding(n_h, d)
        self.proj_c = nn.Linear(C_feat.shape[1], d)
        self.proj_t = nn.Linear(T_feat.shape[1], d)
        self.l1 = RelConv(d, d, n_rel)
        self.l2 = RelConv(d, d, n_rel)
        self.drop = nn.Dropout(dropout)
        self.rel = nn.Parameter(torch.randn(d) * 0.01)
    def encode(self, es):
        x = torch.cat([self.herb_emb.weight, F.relu(self.proj_c(C_feat)), F.relu(self.proj_t(T_feat))], dim=0)
        x = self.drop(F.relu(self.l1(x, es)))
        return self.l2(x, es)
    def score_ct(self, Z, c, t):
        return (Z[n_h + c] * self.rel * Z[n_h + n_c + t]).sum(-1)
    def scores_all(self, Z):
        return (Z[n_h:n_h+n_c] * self.rel) @ Z[n_h+n_c:].t()

#### split, sampling, metric helpers

In [7]:
def split_eh(rng):
    p = rng.permutation(ct_all.shape[1])
    a, b = int(0.70 * len(p)), int(0.85 * len(p))
    return ct_all[:, p[:a]], ct_all[:, p[a:b]], ct_all[:, p[b:]]

def split_cd(rng):
    cs = rng.permutation(np.unique(ct_all[0]))
    a, b = int(0.70 * len(cs)), int(0.85 * len(cs))
    parts = [cs[:a], cs[a:b], cs[b:]]
    return [ct_all[:, np.isin(ct_all[0], x)] for x in parts], parts

def sample_neg_pairs(rng, n, cmps=None):
    pool = neg_key if cmps is None else neg_key[np.isin(neg_key // n_t, cmps)]
    k = rng.choice(pool, size=min(n, len(pool)), replace=False)
    return np.stack([k // n_t, k % n_t])

def tt(a):
    return torch.from_numpy(np.asarray(a)).long().to(device)

def make_esets(ct_tr, parts):
    es = []
    for g in (hc_parts if parts else [hc]):
        h, c = tt(g[0]), tt(g[1] + n_h)
        es += [(h, c), (c, h)]
    cc, t = tt(ct_tr[0] + n_h), tt(ct_tr[1] + n_h + n_c)
    es += [(cc, t), (t, cc)]
    return es

def pr_auc(sp, sn):
    y = np.r_[np.ones(len(sp)), np.zeros(len(sn))]
    return float(average_precision_score(y, np.r_[sp, sn]))

def rank_metrics(r):
    r = np.asarray(r, dtype=np.float64)
    return {'Hits@10': float((r <= 10).mean()), 'Hits@50': float((r <= 50).mean()), 'MRR': float((1 / r).mean())}

#### train one model for a config

In [8]:
def train(cfg, ct_tr, vp, vn, seed):
    torch.manual_seed(seed)
    es = make_esets(ct_tr, cfg['parts'])
    m = Model(len(es), cfg['d'], cfg['dropout']).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    tr, vp_t, vn_t = tt(ct_tr), tt(vp), tt(vn)

    if cfg['ev_weight']:
        w = torch.from_numpy(src_w[ct_tr[0] * n_t + ct_tr[1]]).to(device)
    else:
        w = torch.ones(tr.shape[1], device=device)
    w = w / w.mean()
    nn_ = cfg['n_neg']

    best, wait, best_ep, state = -1, 0, 0, None
    for ep in range(1, EPOCHS + 1):
        m.train()
        perm = torch.randperm(tr.shape[1], device=device)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i+BATCH]
            Z = m.encode(es)
            c_b = tr[0, b]
            sp = m.score_ct(Z, c_b, tr[1, b])
            sn = m.score_ct(Z, c_b[:, None].expand(-1, nn_), torch.randint(0, n_t, (len(b), nn_), device=device))
            loss = -(w[b][:, None] * F.logsigmoid(sp[:, None] - sn)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        if ep % 2 == 0:
            m.eval()
            with torch.no_grad():
                Z = m.encode(es)
                pr = pr_auc(m.score_ct(Z, vp_t[0], vp_t[1]).cpu().numpy(), m.score_ct(Z, vn_t[0], vn_t[1]).cpu().numpy())
            if pr > best:
                best, wait, best_ep = pr, 0, ep
                state = {k: v.clone() for k, v in m.state_dict().items()}
            else:
                wait += 1
                if wait >= PATIENCE:
                    break

    m.load_state_dict(state)
    m.eval()
    with torch.no_grad():
        Z = m.encode(es)
    return m, Z, best, best_ep

#### eval

In [9]:
@torch.no_grad()
def eval_ct(m, Z, te, tn):
    S = m.scores_all(Z)
    c, t = tt(te[0]), tt(te[1])
    s_true = S[c, t]
    higher = S[c] > s_true[:, None]
    filt = 1 + (higher & ~known[c]).sum(1)
    sn = S[tt(tn[0]), tt(tn[1])]
    out = {'PR-AUC': pr_auc(s_true.cpu().numpy(), sn.cpu().numpy())}
    out.update(rank_metrics(filt.cpu().numpy()))
    return out

@torch.no_grad()
def eval_ht(m, Z, ct_tr, ct_va, ct_te):
    S = m.scores_all(Z)
    HS = torch.stack([S[c].max(0).values for c in herb_cmps]).cpu().numpy()
    seen = np.union1d(ht_keys(ct_tr), ht_keys(ct_va))
    test = np.setdiff1d(ht_keys(ct_te), seen)
    flat = HS.ravel()
    out = {'PR-AUC': pr_auc(flat[test], flat[ht_neg])}
    h, t = test // n_t, test % n_t
    filt = 1 + ((HS[h] > HS[h, t][:, None]) & ~ht_known[h]).sum(1)
    out.update(rank_metrics(filt))
    return out

### stage 1: change one thing at a time, pick on val PR-AUC (edge hiding, 3 folds, test not touched)

In [10]:
BASE = dict(lr=1e-3, d=128, dropout=0.2, wd=1e-5, n_neg=1, parts=False, ev_weight=False)
TRIALS = [
    ('baseline',        {}),
    ('n_neg=5',         {'n_neg': 5}),
    ('n_neg=10',        {'n_neg': 10}),
    ('lr=5e-4',         {'lr': 5e-4}),
    ('dropout=0.4',     {'dropout': 0.4}),
    ('d=64',            {'d': 64}),
    ('wd=1e-4',         {'wd': 1e-4}),
    ('plant_part rels', {'parts': True}),
    ('evidence weight', {'ev_weight': True}),
]

# same rng sequence as 05_cv so the val sets match its folds
tune_data = []
for k in range(TUNE_FOLDS):
    rng = np.random.default_rng(100 + k)
    tr, va, te = split_eh(rng)
    vn = sample_neg_pairs(rng, 10 * va.shape[1])
    tune_data.append((tr, va, vn))

def run_trial(name, cfg):
    out = []
    for k, (tr, va, vn) in enumerate(tune_data):
        t0 = time.time()
        _, _, v, ep = train(cfg, tr, va, vn, seed=k)
        out.append({'trial': name, 'fold': k, 'val PR-AUC': v, 'best_ep': ep, 'sec': round(time.time() - t0, 1)})
    print(f"{name:18s} val PR-AUC {np.mean([o['val PR-AUC'] for o in out]):.4f} +/- {np.std([o['val PR-AUC'] for o in out]):.4f}")
    return out

log = []
for name, ch in TRIALS:
    log += run_trial(name, {**BASE, **ch})
log = pd.DataFrame(log)
log.to_csv(f'{OUT}/tune_stage1.csv', index=False)

baseline           val PR-AUC 0.5026 +/- 0.0587
n_neg=5            val PR-AUC 0.4523 +/- 0.0364
n_neg=10           val PR-AUC 0.4684 +/- 0.0853
lr=5e-4            val PR-AUC 0.4291 +/- 0.1090
dropout=0.4        val PR-AUC 0.4705 +/- 0.0298
d=64               val PR-AUC 0.4129 +/- 0.0582
wd=1e-4            val PR-AUC 0.4824 +/- 0.0770
plant_part rels    val PR-AUC 0.4946 +/- 0.0333
evidence weight    val PR-AUC 0.4884 +/- 0.0306


In [11]:
res = log.groupby('trial')['val PR-AUC'].agg(['mean','std']).sort_values('mean', ascending=False)
base = res.loc['baseline', 'mean']
res['gain'] = res['mean'] - base
res.round(4)

,mean,std,gain
trial,,,
baseline,0.5026,0.0719,0.0000
plant_part rels,0.4946,0.0408,-0.0081
evidence weight,0.4884,0.0374,-0.0142
wd=1e-4,0.4824,0.0943,-0.0203
dropout=0.4,0.4705,0.0365,-0.0321
n_neg=10,0.4684,0.1044,-0.0343
n_neg=5,0.4523,0.0446,-0.0504
lr=5e-4,0.4291,0.1336,-0.0735
d=64,0.4129,0.0712,-0.0897


### stage 2: combine the changes that helped

In [12]:
MIN_GAIN = 0.01   # smaller than this is inside fold noise
changes = dict(TRIALS)
combo = {}
for name in res.index:
    if name == 'baseline' or res.loc[name, 'gain'] < MIN_GAIN:
        continue
    for key, val in changes[name].items():
        if key not in combo:
            combo[key] = val
print('combined changes:', combo)

if combo:
    comb_log = pd.DataFrame(run_trial('combined', {**BASE, **combo}))
    comb_mean = comb_log['val PR-AUC'].mean()
    best_single = res.drop('baseline')['mean'].idxmax()
    if comb_mean >= res.loc[best_single, 'mean']:
        BEST = {**BASE, **combo}
    else:
        BEST = {**BASE, **changes[best_single]}
    log = pd.concat([log, comb_log])
    log.to_csv(f'{OUT}/tune_stage1.csv', index=False)
else:
    BEST = dict(BASE)

json.dump(BEST, open(f'{OUT}/best_config.json', 'w'), indent=1)
print('BEST', BEST)

combined changes: {}
BEST {'lr': 0.001, 'd': 128, 'dropout': 0.2, 'wd': 1e-05, 'n_neg': 1, 'parts': False, 'ev_weight': False}


### stage 3: 5 fold cv with BEST on test sets, compare to baseline from 05_cv

In [13]:
rows = []
t0 = time.time()
for k in range(K):
    rng = np.random.default_rng(100 + k)

    tr, va, te = split_eh(rng)
    vn = sample_neg_pairs(rng, 10 * va.shape[1])
    tn = sample_neg_pairs(rng, 50 * te.shape[1])
    m, Z, v, ep = train(BEST, tr, va, vn, seed=k)
    r = {'fold': k, 'split': 'edge_hiding', 'val PR-AUC': v, 'best_ep': ep}
    r.update({f'CT {a}': b for a, b in eval_ct(m, Z, te, tn).items()})
    r.update({f'HT {a}': b for a, b in eval_ht(m, Z, tr, va, te).items()})
    rows.append(r)
    print(f"fold {k} EH  ct prauc {r['CT PR-AUC']:.4f}  ct h@10 {r['CT Hits@10']:.4f}  ht h@10 {r['HT Hits@10']:.4f}  {time.time()-t0:.0f}s")

    (tr, va, te), parts = split_cd(rng)
    vn = sample_neg_pairs(rng, 10 * va.shape[1], parts[1])
    tn = sample_neg_pairs(rng, 50 * te.shape[1], parts[2])
    m, Z, v, ep = train(BEST, tr, va, vn, seed=k)
    r = {'fold': k, 'split': 'compound_disjoint', 'val PR-AUC': v, 'best_ep': ep}
    r.update({f'CT {a}': b for a, b in eval_ct(m, Z, te, tn).items()})
    r.update({f'HT {a}': b for a, b in eval_ht(m, Z, tr, va, te).items()})
    rows.append(r)
    print(f"fold {k} CD  ct prauc {r['CT PR-AUC']:.4f}  ct h@10 {r['CT Hits@10']:.4f}  ht h@10 {r['HT Hits@10']:.4f}  {time.time()-t0:.0f}s")

tuned = pd.DataFrame(rows)
tuned.to_csv(f'{OUT}/tune_cv_folds.csv', index=False)

fold 0 EH  ct prauc 0.2953  ct h@10 0.2974  ht h@10 0.0878  15s
fold 0 CD  ct prauc 0.0642  ct h@10 0.0922  ht h@10 0.0635  48s
fold 1 EH  ct prauc 0.1815  ct h@10 0.2188  ht h@10 0.0841  59s
fold 1 CD  ct prauc 0.0930  ct h@10 0.1328  ht h@10 0.0715  84s
fold 2 EH  ct prauc 0.2341  ct h@10 0.3195  ht h@10 0.0902  102s
fold 2 CD  ct prauc 0.0639  ct h@10 0.1084  ht h@10 0.0629  138s
fold 3 EH  ct prauc 0.0992  ct h@10 0.1101  ht h@10 0.0661  149s
fold 3 CD  ct prauc 0.0796  ct h@10 0.1129  ht h@10 0.0488  194s
fold 4 EH  ct prauc 0.1207  ct h@10 0.2588  ht h@10 0.1193  207s
fold 4 CD  ct prauc 0.0638  ct h@10 0.1057  ht h@10 0.0489  220s


In [14]:
cols = ['CT PR-AUC','CT Hits@10','CT Hits@50','CT MRR','HT PR-AUC','HT Hits@10','HT Hits@50','HT MRR']
old = pd.read_csv(f'{OUT}/cv_folds.csv')

out = []
for sp in ['edge_hiding', 'compound_disjoint']:
    a, b = old[old.split == sp][cols], tuned[tuned.split == sp][cols]
    for c in cols:
        out.append({'split': sp, 'metric': c,
                    'baseline': f'{a[c].mean():.4f} +/- {a[c].std():.4f}',
                    'tuned':    f'{b[c].mean():.4f} +/- {b[c].std():.4f}',
                    'change':   round(b[c].mean() - a[c].mean(), 4)})
cmp = pd.DataFrame(out)
cmp.to_csv(f'{OUT}/tune_compare.csv', index=False)
cmp

,split,metric,baseline,tuned,change
0,edge_hiding,CT PR-AUC,0.1754 +/- 0.0715,0.1862 +/- 0.0807,0.0108
1,edge_hiding,CT Hits@10,0.2404 +/- 0.0826,0.2409 +/- 0.0826,0.0006
2,edge_hiding,CT Hits@50,0.5135 +/- 0.0711,0.5093 +/- 0.0651,-0.0042
3,edge_hiding,CT MRR,0.1261 +/- 0.0501,0.1270 +/- 0.0529,0.0008
4,edge_hiding,HT PR-AUC,0.0675 +/- 0.0074,0.0677 +/- 0.0077,0.0001
5,edge_hiding,HT Hits@10,0.0855 +/- 0.0199,0.0895 +/- 0.0192,0.0040
6,edge_hiding,HT Hits@50,0.2421 +/- 0.0411,0.2402 +/- 0.0385,-0.0019
7,edge_hiding,HT MRR,0.0441 +/- 0.0109,0.0445 +/- 0.0118,0.0004
8,compound_disjoint,CT PR-AUC,0.0769 +/- 0.0163,0.0729 +/- 0.0131,-0.0041
9,compound_disjoint,CT Hits@10,0.1115 +/- 0.0182,0.1104 +/- 0.0147,-0.0011
